# Logistic Regression Project Template
## Binary Outcome Prediction with the Generalized Linear Model

**Purpose**: Reusable notebook template for performing logistic regression analysis on binary response variables following best practices from applied statistics (odds, logits, interpretation in context of the model).

**Based on**: Concepts from *Applied Univariate, Bivariate, and Multivariate Statistics Using Python* (Denis, 2021), Chapter 8 — Logistic Regression and the Generalized Linear Model.

**Typical Use Cases**:
- Predicting failure / success (e.g., O-ring blow-by, equipment failure)
- Market direction (Up / Down)
- Classification of binary clinical, behavioral, or business outcomes
- Any mutually exclusive binary event where probability *p* of occurrence is of interest

---

### How to use this template
1. Replace the sample data loading section with your own dataset.
2. Update the target variable name and predictor list.
3. Run cells sequentially. Comments indicate required vs. optional steps.
4. Always interpret coefficients **in the context of the full model**.
5. Convert logits → odds → probabilities for stakeholder communication.

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Optional but recommended
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, accuracy_score
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded successfully.')

## 2. Data Loading & Initial Inspection

Replace this section with your own data source (CSV, database query, API, etc.).

In [ ]:
# ---------------------------------------------------------------
# EXAMPLE: Challenger O-ring data (classic binary logistic example)
# Replace with your own data loading code.
# ---------------------------------------------------------------

data = {
    'oring': [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'temp':  [53, 57, 58, 63, 66, 67, 67, 67, 68, 69, 70, 70, 70, 70, 72, 73, 75, 75, 76, 76, 78, 79, 81]
}
df = pd.DataFrame(data)

# Quick checks
print('Shape:', df.shape)
print('\nTarget distribution:')
print(df['oring'].value_counts())
print('\nFirst rows:')
df.head()

## 3. Exploratory Data Analysis (EDA)

Key questions before modeling:
- Is the response truly binary and mutually exclusive?
- Are there missing values or extreme outliers?
- What is the base rate of the positive class?
- Do continuous predictors show separation or monotonic relationships with the outcome?

In [ ]:
# Target balance
print('Target value counts (normalized):')
print(df['oring'].value_counts(normalize=True))

# Simple count plot
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='oring', data=df, ax=ax[0], palette='Set2')
ax[0].set_title('Binary Outcome Counts')
ax[0].set_xlabel('Outcome (0 = No Event, 1 = Event)')

# Relationship of key continuous predictor with outcome
sns.boxplot(x='oring', y='temp', data=df, ax=ax[1], palette='Set2')
ax[1].set_title('Predictor distribution by Outcome')
plt.tight_layout()
plt.show()

## 4. Model Specification — Simple Logistic Regression

We model the **logit** (natural log of the odds):

$$
\ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x
$$

where $p$ is the probability of the event (coded as 1).

In [ ]:
# Prepare design matrix (add intercept)
y = df['oring']
X = sm.add_constant(df['temp'])   # intercept + temperature

# Fit logistic regression via Maximum Likelihood
model = sm.Logit(y, X)
results = model.fit(disp=0)  # disp=0 suppresses iteration printout

print(results.summary())

### 4.1 Key Output Interpretation Checklist

- **coef**: Change in the **logit** for a one-unit increase in the predictor (holding other variables constant in multiple models).
- **P>|z|**: Wald test p-value for the coefficient.
- **Pseudo R-squ.**: McFadden’s pseudo R² — useful as a relative measure of fit, not comparable to OLS R².
- **LLR p-value**: Likelihood-ratio test of the overall model vs. null model.
- Always state the **context of the model** when interpreting any coefficient.

## 5. From Logits → Odds → Probabilities

In [ ]:
# Extract coefficients
intercept = results.params['const']
coef_temp = results.params['temp']

print(f'Intercept (logit): {intercept:.4f}')
print(f'Temperature coefficient (logit): {coef_temp:.4f}')

# Odds ratio for a 1-degree increase in temperature
odds_ratio = np.exp(coef_temp)
print(f'\nOdds Ratio (exp(coef)): {odds_ratio:.4f}')
print('Interpretation: For each 1°F increase in temperature, the odds of O-ring failure multiply by this factor (holding other predictors constant).')

# Convert a specific predicted logit back to probability
def logit_to_prob(logit):
    return 1 / (1 + np.exp(-logit))

# Example: predicted probability at 53°F (Challenger launch temperature)
pred_logit_53 = intercept + coef_temp * 53
pred_prob_53 = logit_to_prob(pred_logit_53)
print(f'\nPredicted logit at 53°F: {pred_logit_53:.4f}')
print(f'Predicted probability of failure at 53°F: {pred_prob_53:.4f}')

## 6. Multiple Logistic Regression

When multiple predictors are present, **every coefficient is conditional on the other variables remaining in the model**. Changing the set of predictors almost always changes the numerical value of each coefficient.

In [ ]:
# ---------------------------------------------------------------
# EXAMPLE STRUCTURE for multiple predictors
# Replace column names with your actual predictors.
# ---------------------------------------------------------------

# Example placeholder (will fail on Challenger data because only one continuous predictor exists)
# Uncomment and adapt when you have a multi-predictor dataset:

# predictors = ['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']
# X_multi = sm.add_constant(df[predictors])
# y_multi = df['Direction_binary']  # must be 0/1
# model_multi = sm.Logit(y_multi, X_multi).fit(disp=0)
# print(model_multi.summary())

# Odds ratios for all predictors
# odds_ratios = np.exp(model_multi.params)
# print('\nOdds Ratios:\n', odds_ratios)

## 7. Model Diagnostics & Performance (Recommended)

While classic logistic regression focuses on inference, practical projects also evaluate predictive performance.

In [ ]:
# Predicted probabilities for all observations
df['pred_prob'] = results.predict(X)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int)

print('Confusion Matrix (threshold = 0.5):')
print(pd.crosstab(df['oring'], df['pred_class'], rownames=['Actual'], colnames=['Predicted']))

print('\nClassification Report:')
print(classification_report(df['oring'], df['pred_class']))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(df['oring'], df['pred_prob'])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

## 8. Final Interpretation Template (Copy & Adapt)

Use language similar to the following when reporting results:

> "In a logistic regression model predicting [EVENT] from [PREDICTORS], a one-unit increase in [PREDICTOR] was associated with a change of [COEF] in the log-odds of the event (p = [PVALUE]), holding all other variables in the model constant. The corresponding odds ratio is [OR]. At the observed mean levels of the other predictors, this translates to a predicted probability of approximately [PROB]."

**Critical reminder**: Coefficients (and therefore odds ratios) are **model-context dependent**. Never interpret a single coefficient in isolation from the rest of the model specification.

## 9. Project Checklist Summary

- [ ] Response is binary and mutually exclusive
- [ ] Predictors examined for separation / complete separation issues
- [ ] Intercept included via `sm.add_constant`
- [ ] Model fit via MLE (`sm.Logit` or `sm.GLM` with Binomial family)
- [ ] Coefficients interpreted on the **logit** scale first
- [ ] Odds ratios obtained via `np.exp(coef)`
- [ ] Probabilities obtained via inverse logit transformation
- [ ] All interpretations explicitly reference the full model context
- [ ] Diagnostics (confusion matrix, ROC/AUC, residual checks) reviewed
- [ ] Results communicated in plain language for non-technical stakeholders

---
*Template prepared for practical data science workflow. Adapt data sources, variable names, and domain language to your specific project.*